This notebook is supposed to be run on Kaggle Notebooks (as the original dataset is 50GB) with the following dataset as input:
- https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia

It will load the dataset and then select `training_size` and `testing_size` images for each class, resize them to `image_size`, and save them to a new directory structure for training and testing. Finally, it will zip the new directory structure for download.

Futhermore, it will also create a CSV file with the image names, corresponding labels, and split information (train/test) and other metadata.

In [29]:
import os
import kagglehub
import pandas as pd
import multiprocessing
from PIL import Image
from pathlib import Path
from sklearn.datasets import load_files
import random

In [ ]:
base_dir = Path("/kaggle/input/chest-xray-pneumonia/chest_xray")
output_base = Path("/kaggle/working/nih/data")
classes = ['Normal', 'Pneumonia']
splits = ['train', 'test']

training_size = 1000
testing_size = 250
image_size = 256

In [37]:
# Collect all images by class
all_images = {cls: [] for cls in classes}
for split in splits:
    for cls in classes:
        folder = base_dir / split / cls.upper()
        images = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
        all_images[cls].extend(images)

In [ ]:
# Shuffle and sample
random.seed(42)
samples = {}
for cls in classes:
    random.shuffle(all_images[cls])
    samples[cls] = {
        'train': all_images[cls][:training_size],
        'test': all_images[cls][training_size:training_size+testing_size]
    }

In [39]:
# Create output folders
for split in ['train', 'test']:
    for cls in classes:
        output_dir = output_base / split / cls
        output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Resize and save
size = (image_size, image_size)
for cls in classes:
    for split in ['train', 'test']:
        for img_path in samples[cls][split]:
            try:
                img = Image.open(img_path)
                img = img.resize(size)
                dest = output_base / split / cls / img_path.name
                img.save(dest)
            except Exception as e:
                print(f"Failed to process {img_path}: {e}")

print("Processing complete.")


Processing complete.


In [41]:
import shutil
shutil.make_archive(data_output_path, 'zip', "/kaggle/working/nih/")

'/kaggle/working/nih/data.zip'